# SageMaker Fraud Detection Training Pipeline

This notebook allows you to:
1. Create/update the SageMaker training pipeline
2. Start pipeline executions
3. Monitor training progress
4. View training results and metrics
5. Register trained models in Model Registry

**What this notebook does:**
- ✅ Preprocess data from Athena
- ✅ Train XGBoost model with hyperparameters from config
- ✅ Evaluate model performance
- ✅ Apply quality gates (ROC-AUC ≥ 0.70)
- ✅ Register approved models in SageMaker Model Registry


**Next steps after training:**
After the pipeline completes successfully, proceed to `2_deployment.ipynb` to deploy your trained model to an endpoint.

**Environment:** Run this in a SageMaker AI Notebook instance

## 1. Setup and Configuration

In [1]:
! uv pip install --system -e ../

Using Python 3.12.13 environment at: /opt/conda


⠙ Resolving dependencies...                                                     

⠙ mlflow==3.11.1                                                                

⠙ evidently==0.7.21                                                             

⠹ ipykernel==7.2.0                                                              

...

⠸ Preparing packages... (0/1)                                                   

⠼ Preparing packages... (0/1)                                                   

⠴ Preparing packages... (0/1)                                                   

⠦ Preparing packages... (0/1)                                                   

Prepared 1 package in 1.14s
Uninstalled 1 package in 4ms
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 7ms
 ~ sagemaker-automated-drift-and-trend-monitoring==0.1.0 (from file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring)


### Download training data CSV

Downloads the Kaggle credit-card fraud dataset, transforms it to the project's business-friendly schema, and uploads it to S3 at `s3://<data-bucket>/<prefix>/data/predictions/data.csv`. The Athena `training_data` table is **not** touched here — seeding into Athena is owned by the pipeline's first step (`SeedAthenaTrainingData`).

**Idempotent:** if the predictions CSV already exists in S3, this cell is a few-second no-op. Pass `force=True` to always re-download. Re-seeding Athena is also idempotent — the pipeline's seed step skips if the table integrity check passes.


In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so `src.*` imports work
# even when the kernel's CWD is notebooks/ and the editable install
# hasn't been refreshed for this kernel session.
_project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.setup.download_kaggle_dataset import ensure_training_data_downloaded

ensure_training_data_downloaded()  # add force=True to force a full re-download


In [3]:
import os
import sys
import boto3
from sagemaker.core.helper.session_helper import Session
from pathlib import Path
from dotenv import load_dotenv
from src.utils.aws_utils import get_execution_role

# Determine project root and load .env
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
env_path = project_root / '.env'

# Load .env file with override=True to ensure values are loaded
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"✓ Loaded environment from: {env_path}\n")
else:
    print(f"⚠ .env file not found at: {env_path}\n")

# Initialize AWS clients
region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
sagemaker_client = boto3.client('sagemaker', region_name=region)
s3_client = boto3.client('s3', region_name=region)

# Get SageMaker session and role using project's wrapper
# This handles fallback to .env and provides better error messages
role = get_execution_role()

sagemaker_session = Session()
default_bucket = sagemaker_session.default_bucket()

# Display configuration
print(f"AWS Configuration:")
print(f"  Region: {region}")
print(f"  Role: {role}")
print(f"  Default S3 bucket: {default_bucket}")

# MLflow Tracking Configuration
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
mlflow_experiment = os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection')

print(f"\nMLflow Configuration:")
if mlflow_uri:
    print(f"  Tracking URI: {mlflow_uri}")
    print(f"  Experiment: {mlflow_experiment}")
    print(f"\n✓ MLflow tracking is ENABLED")
    print(f"  Training metrics will be logged to MLflow")
    print(f"  View experiments at: SageMaker Console → MLflow")
else:
    print(f"  Tracking URI: Not set in .env")
    print(f"\n⚠ MLflow tracking is NOT configured")
    print(f"  Add MLFLOW_TRACKING_URI to .env file to enable tracking")
    print(f"  Example: MLFLOW_TRACKING_URI=arn:aws:sagemaker:us-east-1:123456789012:mlflow-app/app-XXXXX")

✓ Loaded environment from: /home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/.env

Using SageMaker execution role from environment: arn:aws:iam::146666888814:role/fraud-detection-monitoring-SageMakerExecutionRole
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


AWS Configuration:
  Region: us-west-2
  Role: arn:aws:iam::146666888814:role/fraud-detection-monitoring-SageMakerExecutionRole
  Default S3 bucket: sagemaker-us-west-2-146666888814

MLflow Configuration:
  Tracking URI: arn:aws:sagemaker:us-west-2:146666888814:mlflow-tracking-server/fraud-detection-monitoring-mlflow
  Experiment: credit-card-fraud-detection

✓ MLflow tracking is ENABLED
  Training metrics will be logged to MLflow
  View experiments at: SageMaker Console → MLflow


In [4]:
# Pipeline configuration
PIPELINE_NAME = "fraud-detection-pipeline"
PIPELINE_DESCRIPTION = "Fraud detection ML pipeline: preprocessing, training, evaluation, and model registration"

# Pipeline parameter OVERRIDES for this execution.
# Hyperparameter defaults (MaxDepth, LearningRate, NumBoostRound,
# MinChildWeight, EarlyStoppingRounds) and quality-gate defaults
# (MinRocAuc, MinPrAuc) live in src/config/config.yaml under
# training.xgboost_params and are wired through pipeline.py — leave them
# out here so the YAML stays the single source of truth.
# Override one in PIPELINE_PARAMS only if you want to deviate from YAML
# for a specific run.
PIPELINE_PARAMS = {
    # Data
    'AthenaTable': 'training_data',
    
    # Model Registry
    'ModelApprovalStatus': 'Approved',
    'ModelPackageGroup': 'fraud-detection',
}

print("Pipeline Configuration:")
print(f"  Name: {PIPELINE_NAME}")
print(f"  Description: {PIPELINE_DESCRIPTION}")
print("\nXGBoost hyperparameters: from src/config/config.yaml (training.xgboost_params)")
print("Quality gates: from pipeline.py defaults (MinRocAuc=0.70, MinPrAuc=0.30)")
print("\nExecution-level overrides:")
for key, value in PIPELINE_PARAMS.items():
    print(f"  {key}: {value}")

print("\n" + "="*80)
print("NOTE: This pipeline does NOT include deployment steps.")
print("After training completes, use 2_deployment.ipynb to deploy the model.")
print("="*80)

Pipeline Configuration:
  Name: fraud-detection-pipeline
  Description: Fraud detection ML pipeline: preprocessing, training, evaluation, and model registration

XGBoost hyperparameters: from src/config/config.yaml (training.xgboost_params)
Quality gates: from pipeline.py defaults (MinRocAuc=0.70, MinPrAuc=0.30)

Execution-level overrides:
  AthenaTable: training_data
  ModelApprovalStatus: Approved
  ModelPackageGroup: fraud-detection

NOTE: This pipeline does NOT include deployment steps.
After training completes, use 2_deployment.ipynb to deploy the model.


## 3. Create/Update Pipeline

This will create the pipeline definition in SageMaker using the fixed `train.py` code.

In [5]:
# Optional: Delete pipeline to force code refresh
# Uncomment ONLY if you changed code in src/train_pipeline/
#
# Note: Deleting pipeline does NOT affect Model Registry versions.
# Model versions (v1, v2, v3...) will continue incrementing regardless.
#
# When to uncomment:
#   ✓ You changed pipeline code or steps
#   ✓ Need to force refresh of containerized code
#   ✓ Pipeline definition is corrupted
#
# When NOT needed:
#   ✗ Just running another training (pipeline reuses definition)
#   ✗ Only hyperparameters changed (use PIPELINE_PARAMS)

try:
    sagemaker_client.delete_pipeline(PipelineName=PIPELINE_NAME)
    print("✓ Pipeline deleted")
except sagemaker_client.exceptions.ResourceNotFound:
    print("ℹ Pipeline does not exist")
except Exception as e:
    print(f"⚠ Error deleting pipeline: {e}")

print("ℹ Pipeline deletion skipped")
print("  Tip: Uncomment above only if you changed pipeline code")

⚠ Error deleting pipeline: An error occurred (ValidationException) when calling the DeletePipeline operation: Pipelines with running executions cannot be deleted. Pipeline arn:aws:sagemaker:us-west-2:146666888814:pipeline/fraud-detection-pipeline has 1 running executions.
ℹ Pipeline deletion skipped
  Tip: Uncomment above only if you changed pipeline code


In [6]:
from src.train_pipeline.pipeline import create_fraud_detection_pipeline

print("Creating/updating pipeline...\n")

# Display MLflow integration info
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
if mlflow_uri:
    print("MLflow Integration:")
    print(f"  ✓ Training step will log to: {mlflow_uri}")
    print(f"  ✓ Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    print(f"  ✓ Model registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
    print()

try:
    # Create pipeline builder
    pipeline_builder = create_fraud_detection_pipeline(
        pipeline_name=PIPELINE_NAME,
        region=region,
        role=role
    )
    
    # Upsert pipeline (create if doesn't exist, update if it does)
    # include_deployment=False means this pipeline only trains and registers models
    # Use 2_deployment.ipynb to deploy the trained model
    result = pipeline_builder.upsert_pipeline(
        description=PIPELINE_DESCRIPTION,
        include_deployment=False,  # Training only - no endpoint deployment
        tags=[
            {'Key': 'Project', 'Value': 'FraudDetection'},
            {'Key': 'Environment', 'Value': 'Production'},
            {'Key': 'ManagedBy', 'Value': 'MLflow'},
            {'Key': 'PipelineType', 'Value': 'Training'},
        ]
    )
    
    print("✓ Pipeline created/updated successfully!")
    print(f"\nPipeline ARN: {result['pipeline_arn']}")
    print(f"\nPipeline steps:")
    print(f"  1. Preprocess - Read from Athena, validate, split data")
    print(f"  2. Train - XGBoost model training")
    print(f"  3. Evaluate - Compute metrics and apply quality gates")
    print(f"  4. Register - Register approved models in Model Registry")
    print(f"\n⚠️  This pipeline does NOT include deployment.")
    print(f"   After training, use 2_deployment.ipynb to deploy the model.")
    
except Exception as e:
    print(f"✗ Failed to create pipeline: {e}")
    import traceback
    traceback.print_exc()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 from src.train_pipeline.pipeline import create_fraud_detection_pipeline                     │
│    2                                                                                             │
│    3 print("Creating/updating pipeline...\n")                                                    │
│    4                                                                                             │
│                                                                                                  │
│ /home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/s │
│ rc/train_pipeline/pipeline.py:48 in <module>                                                     │
│                                                                                                  │
│     45 from sagemaker.core.inputs import TrainingInput, CreateModelInput                         │
│     46                                                                                           │
│     47 # Model (ModelBuilder replaces the old sagemaker.model.Model in v3)                       │
│ ❱   48 from sagemaker.serve.model_builder import ModelBuilder                                    │
│     49 from sagemaker.core.model_metrics import MetricsSource, ModelMetrics                      │
│     50                                                                                           │
│     51 # Workflow parameters                                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/serve/__init__.py:30 in <module>               │
│                                                                                                  │
│   27 # Match V2's imports to ensure same priming behavior                                        │
│   28 from sagemaker.serve.spec.inference_spec import InferenceSpec                               │
│   29 from sagemaker.serve.utils.types import ModelServer                                         │
│ ❱ 30 from sagemaker.serve.model_builder import ModelBuilder                                      │
│   31                                                                                             │
│   32 __all__ = ["InferenceSpec", "ModelServer", "ModelBuilder"]                                  │
│   33                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py:85 in <module>          │
│                                                                                                  │
│     82 from sagemaker.serve.validations.check_image_uri import is_1p_image_uri                   │
│     83 from sagemaker.core.inference_config import ResourceRequirements                          │
│     84 from sagemaker.serve.inference_recommendation_mixin import _InferenceRecommenderMixin     │
│ ❱   85 from sagemaker.serve.model_builder_utils import _ModelBuilderUtils                        │
│     86 from sagemaker.serve.model_builder_servers import _ModelBuilderServers                    │
│     87 from sagemaker.serve.validations.optimization import _validate_optimization_configuratio  │
│     88 from sagemaker.core.enums import Tag                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/serve/mod

## 4. Start Pipeline Execution

This will start a new execution of the pipeline with the fixed training code.

In [ ]:
# Generate execution name with timestamp
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
execution_name = f"{PIPELINE_NAME}-{timestamp}"

print(f"Starting pipeline execution: {execution_name}\n")

# Format parameters for SageMaker
pipeline_parameters = [
    {'Name': key, 'Value': str(value)}
    for key, value in PIPELINE_PARAMS.items()
]

try:
    response = sagemaker_client.start_pipeline_execution(
        PipelineName=PIPELINE_NAME,
        PipelineExecutionDisplayName=execution_name,
        PipelineParameters=pipeline_parameters
    )
    
    execution_arn = response['PipelineExecutionArn']
    
    print("✓ Pipeline execution started successfully!")
    print(f"\nExecution Details:")
    print(f"  ARN: {execution_arn}")
    print(f"  Name: {execution_name}")
    
    # MLflow logging information
    mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
    if mlflow_uri:
        print(f"\nMLflow Tracking:")
        print(f"  📊 Training metrics are being logged to MLflow")
        print(f"  📈 Tracking Server: {mlflow_uri}")
        print(f"  🔬 Experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
        print(f"  📦 Model Registry: {os.getenv('MLFLOW_MODEL_NAME', 'xgboost-fraud-detector')}")
        print(f"\n  View in MLflow UI:")
        print(f"    1. Open SageMaker Console → MLFlow")
        print(f"    2. Select your MLflow app")
        print(f"    3. Find experiment: {os.getenv('MLFLOW_EXPERIMENT_NAME', 'credit-card-fraud-detection-training')}")
    else:
        print(f"\n⚠ MLflow tracking not configured")
    
    print(f"\nYou can monitor this execution in the next cell")
    
    # Store for monitoring
    CURRENT_EXECUTION_ARN = execution_arn
    
    print("\n" + "="*80)
    print("NEXT STEPS AFTER PIPELINE COMPLETES:")
    print("="*80)
    print("1. Wait for pipeline to complete successfully")
    print("2. Model will be registered in SageMaker Model Registry")
    print("3. Open 2_deployment.ipynb to deploy the trained model")
    print("4. Select the registered model and deploy to endpoint")
    print("="*80)
    
except Exception as e:
    print(f"✗ Failed to start execution: {e}")
    import traceback
    traceback.print_exc()

## 5. Monitor Pipeline Execution

Watch the pipeline execution in real-time.

In [ ]:
def get_actual_metrics(execution_arn):
    """Get actual metrics from the evaluation report."""
    import boto3
    import json

    # Use the region variable from the notebook scope
    sagemaker_client = boto3.client('sagemaker', region_name=region)
    s3_client = boto3.client('s3', region_name=region)

    # Get execution steps
    response = sagemaker_client.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn
    )

    steps = response.get('PipelineExecutionSteps', [])

    # Find evaluation step
    for step in steps:
        if 'EvaluateModel' in step['StepName']:
            if 'Metadata' in step and 'ProcessingJob' in step['Metadata']:
                job_name = step['Metadata']['ProcessingJob']['Arn'].split('/')[-1]

                # Get processing job details
                response = sagemaker_client.describe_processing_job(
                    ProcessingJobName=job_name
                )

                # Find evaluation output
                for output in response['ProcessingOutputConfig']['Outputs']:
                    if output['OutputName'] == 'evaluation':
                        s3_uri = output['S3Output']['S3Uri']

                        # Parse S3 URI
                        parts = s3_uri.replace('s3://', '').split('/', 1)
                        bucket = parts[0]
                        key = parts[1] + '/evaluation.json'

                        print(f"Fetching evaluation report from:")
                        print(f"  s3://{bucket}/{key}\n")

                        try:
                            # Download evaluation report
                            obj = s3_client.get_object(Bucket=bucket, Key=key)
                            property_data = json.loads(obj['Body'].read())

                            print("="*80)
                            print("ACTUAL MODEL PERFORMANCE:")
                            print("="*80)

                            metrics = property_data.get('binary_classification_metrics', {})
                            for metric_name, metric_value in metrics.items():
                                value = metric_value.get('value', 'N/A')
                                print(f"  {metric_name.upper()}: {value:.4f}" if isinstance(value, float) else f"  {metric_name.upper()}: {value}")

                            print("="*80)

                            # Also check full report
                            key_full = parts[1] + '/evaluation_report.json'
                            try:
                                obj_full = s3_client.get_object(Bucket=bucket, Key=key_full)
                                full_report = json.loads(obj_full['Body'].read())

                                print("\nQUALITY GATES:")
                                print("="*80)
                                qg = full_report.get('quality_gates', {})
                                print(f"  Passed: {qg.get('passed', 'N/A')}")

                                for check in qg.get('checks', []):
                                    status = "✓" if check['passed'] else "✗"
                                    print(f"  {status} {check['metric'].upper()}: {check['value']:.4f} (threshold: {check['threshold']:.4f})")

                                if qg.get('failures'):
                                    print(f"\n  Failures: {', '.join(qg['failures'])}")

                                print("="*80)

                            except Exception as e:
                                print(f"Could not read full report: {e}")

                            return property_data

                        except Exception as e:
                            print(f"✗ Could not read evaluation report: {e}")

# Run for current execution
if 'CURRENT_EXECUTION_ARN' in locals():
    get_actual_metrics(CURRENT_EXECUTION_ARN)
else:
    print("No execution found. Set CURRENT_EXECUTION_ARN first.")

## 6. Utility Functions

Additional helper functions for pipeline management.

In [ ]:
def stop_execution(execution_arn):
    """Stop a running pipeline execution."""
    try:
        response = sagemaker_client.stop_pipeline_execution(
            PipelineExecutionArn=execution_arn
        )
        print(f"✓ Stopped execution: {execution_arn}")
        return response
    except Exception as e:
        print(f"✗ Failed to stop execution: {e}")
        return None

def delete_pipeline(pipeline_name):
    """Delete a pipeline (use with caution!)."""
    try:
        response = sagemaker_client.delete_pipeline(
            PipelineName=pipeline_name
        )
        print(f"✓ Deleted pipeline: {pipeline_name}")
        return response
    except sagemaker_client.exceptions.ResourceNotFound:
        print(f"ℹ Pipeline '{pipeline_name}' does not exist (already deleted or never created)")
        return None
    except Exception as e:
        print(f"✗ Failed to delete pipeline: {e}")
        return None

def get_pipeline_definition(pipeline_name):
    """Get pipeline definition JSON."""
    try:
        response = sagemaker_client.describe_pipeline(
            PipelineName=pipeline_name
        )
        definition = json.loads(response['PipelineDefinition'])
        return definition
    except Exception as e:
        print(f"✗ Failed to get pipeline definition: {e}")
        return None

print("Utility functions loaded:")
print("  - stop_execution(execution_arn)")
print("  - delete_pipeline(pipeline_name)")
print("  - get_pipeline_definition(pipeline_name)")

In [ ]:
import boto3, pandas as pd, io, time

athena_client = boto3.client('athena', region_name=region)
s3_client = boto3.client('s3', region_name=region)

response = athena_client.start_query_execution(
    QueryString="SELECT * FROM training_data LIMIT 3",
    QueryExecutionContext={'Database': 'fraud_detection'},
    ResultConfiguration={'OutputLocation': f's3://{default_bucket}/athena-query-results/'}
)

time.sleep(10)

response = athena_client.get_query_execution(QueryExecutionId=response['QueryExecutionId'])
output_uri = response['QueryExecution']['ResultConfiguration']['OutputLocation']
bucket = output_uri.replace('s3://', '').split('/')[0]
key = '/'.join(output_uri.replace('s3://', '').split('/')[1:])

obj = s3_client.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Athena columns: {list(df.columns)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nFirst 2 rows:")
print(df.head(2))

## 7. Quick Reference

### Common Tasks

**Start a new execution with custom parameters:**
```python
PIPELINE_PARAMS['MinRocAuc'] = '0.90'
PIPELINE_PARAMS['EndpointName'] = 'fraud-detector-v2'
# Then run cell 4 to start execution
```

**Stop a running execution:**
```python
stop_execution(CURRENT_EXECUTION_ARN)
```

**Monitor a specific execution:**
```python
execution_arn = 'arn:aws:sagemaker:us-east-1:123456789012:pipeline/fraud-detection-pipeline/execution/xyz'
monitor_execution(execution_arn)
```

**View CloudWatch logs:**
- Go to CloudWatch Console
- Navigate to Log Groups
- Search for `/aws/sagemaker/ProcessingJobs` or `/aws/sagemaker/TrainingJobs`

### Expected Results

With `train.py`, you should see:
- ✓ **ROC-AUC > 0.85** (likely 0.90-0.95)
- ✓ **PR-AUC > 0.50** (likely 0.60-0.80)
- ✓ **True Positives > 0** (model detects fraud)
- ✓ **Quality gates pass**
- ✓ **Pipeline completes successfully**


### Troubleshooting

**Pipeline creation fails:**
- Check that `.env` file is properly configured
- Verify SageMaker execution role has necessary permissions
- Check S3 bucket access

**Training step fails:**
- Check CloudWatch logs for the training job
- Verify input data path is correct
- Check that preprocessing completed successfully

**Evaluation fails quality gates:**
- Check evaluation metrics in cell 6
- If ROC-AUC is still 0.50, verify the pipeline picked up the fixed code
- Consider adjusting quality gate thresholds in PIPELINE_PARAMS